# E25 — GPU pilot (HandReach / MAE-B / seed 0)
Run in order on a GPU runtime with internet enabled. This is one official-sized cell, not the 24-cell benchmark or a SOTA result. Uses the existing CUDA PyTorch build; no CUDA reinstall. Upstream and E25 source revisions are pinned. Population images are rendered but not saved; only their physical outcomes are needed. The runtime is not the exact published environment.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
from importlib.metadata import version

ROOT = Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content') / 'e25_gpu_v3'
ROOT.mkdir(parents=True, exist_ok=True)
UP = ROOT / 'counterfactual-WM'
BUNDLE = ROOT / 'e25'
BUNDLE.mkdir(exist_ok=True)
PY = sys.executable
PIN = '5c84bba92b2ff19d41175070a61238fa1396d10d'
E25_PIN = 'e6c3dff96bd9a0592ba91e908c757282f3c74a4b'
ENV = dict(os.environ, CWM_OUTPUTS=str(ROOT/'outputs'), MUJOCO_GL='egl',
           PYOPENGL_PLATFORM='egl', SDL_VIDEODRIVER='dummy', WANDB_MODE='disabled',
           PYTHONUNBUFFERED='1', HF_HOME=str(ROOT/'hf_cache'))
def shell(*args, cwd=ROOT, capture=False):
    args = list(map(str, args))
    print('>', ' '.join(args), flush=True)
    return subprocess.run(args, cwd=cwd, env=ENV, check=True, text=True,
                          stdout=subprocess.PIPE if capture else None).stdout

def py(*args, capture=False):
    return shell(PY, *args, cwd=UP, capture=capture)

shell('nvidia-smi')
shell(PY, '-c', "import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(), 'CPU-only torch or no GPU: select GPU and use its CUDA-enabled Python environment'; print(torch.cuda.get_device_name(0)); print(torch.ones(4,device='cuda').sum().item())")
print('Free disk GiB:', round(shutil.disk_usage(ROOT).free/2**30, 1))

In [ ]:
# Download only the selected source revision and install dependencies for HandReach.
if not UP.exists():
    shell('git', 'clone', 'https://github.com/galilai-group/counterfactual-WM', UP)
shell('git', 'checkout', '--detach', PIN, cwd=UP)
assert shell('git', 'rev-parse', 'HEAD', cwd=UP, capture=True).strip() == PIN
constraints = ROOT/'constraints.txt'
constraints.write_text('\n'.join(f'{p}=={version(p)}' for p in ('torch','torchvision','numpy'))+'\n')
py('-m','pip','install','-q','-c',constraints,
   'mujoco==3.11.0','gymnasium==1.2.3','gymnasium-robotics==1.4.2',
   'timm==1.0.26','transformers==5.4.0',
   'stable-pretraining @ git+https://github.com/galilai-group/stable-pretraining@ab836bf699a2be712dfcf980a5eb70d36391e876')
py('-m','pip','install','-q','--no-deps','-e',UP)
py('-c', "import torch; from cwm.model import WorldModel; from cwm import train; assert torch.cuda.is_available(); print('Imports and GPU OK')")

In [ ]:
# Fetch the committed E25 sources. Cache the posterior solve once per observation.
import urllib.request
for name in ('e25.py','run_e25.py','score_e25.py','test_e25.py'):
    url = f'https://raw.githubusercontent.com/shekhawatxyz/fastai-2022/{E25_PIN}/experiments/recursive_world/e25/{name}'
    with urllib.request.urlopen(url, timeout=60) as r:
        (BUNDLE/name).write_bytes(r.read())
p = BUNDLE/'e25.py'
s = p.read_text()
def replace_once(s, old, new):
    assert s.count(old) == 1, f'Unexpected source revision: {old}'
    return s.replace(old, new, 1)
s = replace_once(s, 'self.raw_last = torch.zeros_like(self.sigma)',
                 'self.raw_last = torch.zeros_like(self.sigma)\n        self._cached = None')
s = replace_once(s, 'action = action.reshape(-1)',
                 'self._cached = None\n        action = action.reshape(-1)')
s = replace_once(s, 'p = self.sigma.new_zeros(2*self.a+self.d)\n        p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\n        return p',
                 'if self._cached is None:\n            p = self.sigma.new_zeros(2*self.a+self.d)\n            p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\n            self._cached = p\n        return self._cached')
p.write_text(s)
# Resume complete records rather than restarting the whole expensive evaluation.
p = BUNDLE/'run_e25.py'; s = p.read_text()
s = replace_once(s, "with args.out.open('x') as f: # no overwrite or accidental duplicate episodes",
"""expected_keys = {(i,c,a) for i in range(100*args.seed,100*args.seed+args.count)
                         for c in design['conditions'] for a in arms}
        done = set()
        if args.out.exists():
            raw = args.out.read_bytes()
            complete = raw[:raw.rfind(b'\n')+1]
            for line in complete.splitlines():
                r = json.loads(line)
                key = (r['episode'],r['condition'],r['arm'])
                if (key not in expected_keys or key in done or r['mode'] != mode
                    or r['seed_base'] != design['seed_base']
                    or any(r.get(k) != v for k,v in tag.items())
                    or r['noise'] != list(map(float,noise)) or r['alpha'] != alpha):
                    raise ValueError('Incompatible previous output: use a fresh directory')
                done.add(key)
            if complete != raw: args.out.write_bytes(complete)
        with args.out.open('a') as f:""".replace("b'\n'", "b'\\n'"))
s = replace_once(s, "r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)",
                 "if (episode,condition,arm) in done: continue\n                        r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)")
s = replace_once(s, "f.write(json.dumps(r)+'\\n');f.flush()", "f.write(json.dumps(r)+'\\n');f.flush()\n                print(mode, episode+1, '/ 100', flush=True)")
compile(s,str(p),'exec'); p.write_text(s)
py(BUNDLE/'test_e25.py')

In [ ]:
# Fail early if MuJoCo EGL or the pinned encoder cannot run.
py('-c', """
from cwm import registry, encoders
from cwm.sims import make
import numpy as np, torch
sim=make(registry.simulator('hand','population'))
try:
    sim.reset(2201000000)
    frame=sim.render()
    print('Rendered', frame.shape)
    assert frame.shape == (224,224,3)
finally: sim.close()
model=encoders.load('mae_cls','cuda')
z=encoders.encode(model,np.asarray([frame]),batch_size=1)
assert torch.isfinite(z).all()
print('Pinned encoder OK',z.shape)
""")
# Capture the actual installed versions for reproducibility.
(ROOT/'environment.txt').write_text(py('-m','pip','freeze',capture=True))

In [ ]:
# Generate the original factual training data, then fit the original frozen model.
py('experiments/2_data/collect.py','hand')
py('experiments/2_data/encode.py','hand','mae_cls','--device','cuda')
py('experiments/3_train/readouts.py','hand','mae_cls','--index',0,'--device','cuda')
script='experiments/3_train/predictors.py'
index=py(script,'planning','--list',capture=True).splitlines().index('hand_mae_cls_h1_s0')
py(script,'planning','--index',index,'--device','cuda')

In [ ]:
# Preserve all population render calls and simulator branches, but discard image writes.
# The planner consumes only branches.npz, not population frames.npy.
py('-c', """
from pathlib import Path
import numpy as np
from cwm import registry
from cwm.sims import population
out=registry.data('hand','population')
if not (out/'results.json').exists():
    import shutil
    shutil.rmtree(out, ignore_errors=True)
    original=np.lib.format.open_memmap
    class FrameSink:
        def __setitem__(self,key,value): pass
        def flush(self): pass
    def open_memmap(filename,*args,**kwargs):
        if Path(filename).resolve()==(out/'frames.npy').resolve(): return FrameSink()
        return original(filename,*args,**kwargs)
    np.lib.format.open_memmap=open_memmap
    try: population.run(registry.simulator('hand','population'),out)
    finally: np.lib.format.open_memmap=original
assert (out/'branches.npz').exists()
print('Population complete')
""")

In [ ]:
# Compare E25's two unchanged arms with the upstream Planner on disjoint smoke seeds.
# Only implementation parity is checked here, not E25's research hypothesis.
py('-c', f"""
import sys, importlib.util, numpy as np
from pathlib import Path
sys.path.insert(0,{str(BUNDLE)!r})
from cwm import registry, io
from cwm.model import WorldModel
from cwm.sims.worker import Remote
from run_e25 import run_episode
spec=importlib.util.spec_from_file_location('planning', 'experiments/4_evaluate/planning.py')
up=importlib.util.module_from_spec(spec); spec.loader.exec_module(up)
d=io.read_json(registry.CONFIGS/'planning/replan.json'); d['seed_base']+=2000000
fit=registry.OUT/'fits/hand_mae_cls_h1_s0/fit.pt'
m=WorldModel('mae_cls',fit,registry.readout('hand','mae_cls','state'),'cuda')
sigma=up.sigma_of('hand'); idx=registry.dataset('hand')['coordinates']
sim=Remote(registry.simulator('hand','population'),up.sim_command('hand'))
try:
    planner=up.Planner(sim,m,d,idx,sigma)
    for condition in d['conditions']:
        for arm in d['arms']:
            goal=sim('fix',np.random.default_rng(np.random.SeedSequence([d['seed_base'],0,14])).uniform(-planner.limit,planner.limit,m.a))
            ref=planner.replan(0,condition,arm,goal)
            row=run_episode(up,sim,m,d,idx,sigma,0,condition,arm,np.ones_like(sigma),0.)
            np.testing.assert_allclose(row['distances'],ref['distances'],atol=1e-5,rtol=0)
    print('Six smoke baseline parity checks passed')
finally: sim.close()
""")

In [ ]:
# Official-size pilot. Upstream baseline runs only replan, not unrelated retry tasks.
py('-c', """
import importlib.util
from pathlib import Path
from cwm import io
spec=importlib.util.spec_from_file_location('planning','experiments/4_evaluate/planning.py')
up=importlib.util.module_from_spec(spec); spec.loader.exec_module(up)
original=io.read_json
def read_json(path):
    d=original(path)
    if Path(path).name=='replan.json': d['cells']['hand']['tasks']=['replan']
    return d
io.read_json=read_json
up.learned('replan','hand','mae_cls',0,'cuda')
""")
cal=ROOT/'calibration/hand_mae_cls_s0.jsonl'
test=ROOT/'test/hand_mae_cls_s0.jsonl'
base=[BUNDLE/'run_e25.py','--upstream',UP,'--dataset','hand','--encoder','mae_cls','--seed',0,'--device','cuda']
py(*base,'--out',cal)
py(*base,'--calibration',cal,'--out',test)
report=py(BUNDLE/'score_e25.py',test,'--scope','pilot','--upstream-logs',ROOT/'outputs/planning/replan',capture=True)
(ROOT/'score.json').write_text(report)
print(report)

In [ ]:
# Export the evidence (also works after partial execution).
import zipfile
archive=ROOT/'e25_evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (BUNDLE,ROOT/'calibration',ROOT/'test',ROOT/'outputs/planning/replan'):
        if folder.exists():
            for p in folder.rglob('*'):
                if p.is_file() and p.suffix in ('.py','.jsonl','.json'):
                    z.write(p,p.relative_to(ROOT))
    for p in (ROOT/'environment.txt',ROOT/'score.json'):
        if p.exists(): z.write(p,p.name)
from IPython.display import FileLink, display
print(archive)
display(FileLink(str(archive)))